In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import numpy as np
from sqlalchemy import create_engine

## 1. Get data

In [3]:
username = 'rodrigo'
host = 'localhost'           
port = '5432'               
database = 'futmondo_full_players_info'

engine = create_engine(f'postgresql+psycopg2://{username}@{host}:{port}/{database}')
# query = "SELECT * FROM full_training_data"
query= """SELECT *
    FROM full_training_data;"""

df = pd.read_sql(query, engine)
df.shape

(4677, 39)

In [4]:
df.head(2)

,player_id,name,role,round,home_average,away_average,overall_average,current_price,matches_played,rating,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
0,51ffb2540ac2ec8b0700001e,Dani Rodríguez,centrocampista,13,2.0,0.0,2.000000,1000000,2,1,...,0.0,-0.528376,0.605592,0.0,0.160761,0.0,0.0,0.000000,1,0
1,52013ee178b20d7f07000351,Josan,centrocampista,13,4.4,1.5,3.571429,1000000,7,1,...,4.4,-0.570438,0.555189,0.0,0.150752,0.0,0.0,0.062813,1,0


In [65]:
# df=pd.read_csv('data/final_dataset/futmondo_final_dataset.csv')
# df.shape

## 2. Build the model

In [5]:
# Corrected feature columns
feature_columns = [
    'home_average',
    'away_average', 
    'rating',       # given by futmondo
    'overall_average', 
    'last_2_average',  # Changed from last_3_average
    'current_price',
    'is_home',  # Removed is_home_target (duplicate)
    'match_minus_1',
    'match_minus_2',  # Removed match_minus_3 (not available)
    'matchup_prob_win', 
    'matchup_prob_draw', 
    'matchup_prob_loss', 
    'form_trend', 
    'home_away_diff', 
    'price_per_point',  # Changed from price_vs_max
    # 'price_efficiency',  # Changed from price_volatility
    'recent_momentum', 
    'home_form_interaction',
    'away_form_interaction',
    'location_adjusted_average',  # Added new feature
    'matchup_strength',
    'team_expected_performance', 
    'delantero_matchup_bonus',
    'centrocampista_matchup_bonus', 
    'defensa_matchup_bonus',
    'portero_matchup_bonus', 
    'home_matchup_boost', 
    'difficult_matchup',
    'easy_matchup'
]

# Drop rows where target_points or any feature is NaN
df_clean = df.dropna(subset=['target_points'] + feature_columns)
X = df_clean[feature_columns]
y = df_clean['target_points']

In [6]:
df.isna().sum()

player_id                         0
name                              0
role                              0
round                             0
home_average                     13
away_average                     22
overall_average                  30
current_price                     0
matches_played                    0
rating                            0
team_id                           0
match_minus_1                     0
match_minus_2                     0
last_2_average                    0
target_points                   517
unique_id                         0
team                              0
matchup_prob_win                 96
matchup_prob_draw                96
matchup_prob_loss                96
is_home                          96
opponent                         96
form_trend                       30
home_away_diff                   35
price_per_point                  30
price_efficiency                 30
recent_momentum                   0
home_form_interaction       

In [52]:
df_clean.shape

(4081, 39)

In [53]:
# # Check for NaN values first
# print("NaN values per column:")
# print(X.isna().sum())
# print(f"\nTotal NaN values: {X.isna().sum().sum()}")

# # Option 1: Drop rows with NaN values (if not too many)
# X_clean = X.dropna()
# y_clean = y[X_clean.index]

# print(f"\nRows before: {len(X)}, Rows after: {len(X_clean)}")

In [54]:
# Scale features for better performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

# Ensemble model with multiple algorithms
gb_model = GradientBoostingRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42
)

ridge_model = Ridge(alpha=10.0, random_state=42)

rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features='sqrt',
    random_state=42
)

# Voting ensemble combines predictions from all models
model = VotingRegressor(
    estimators=[
        ('gb', gb_model),
        ('rf', rf_model),
        ('ridge', ridge_model)
    ],
    weights=[2, 2, 1]  # Give more weight to tree-based models
)

# Train the ensemble
model.fit(X_train, y_train)

,estimators,"[('gb', ...), ('rf', ...), ...]"
,weights,"[2, 2, ...]"
,n_jobs,None
,verbose,False
,loss,'squared_error'
,learning_rate,0.05
,n_estimators,150
,subsample,0.8
,criterion,'friedman_mse'
,min_samples_split,10
,min_samples_leaf,4


In [55]:
# ===== PREDICTIONS =====
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# Create predictions dataframe for test set
test_indices = len(y_test)
predictions_df = pd.DataFrame({
    'actual_points': y_test,
    'predicted_points': y_pred_test,
    'error': y_test - y_pred_test,
    'abs_error': np.abs(y_test - y_pred_test)})

In [56]:
# # Add original features (unscaled) for context
# X_test_unscaled = X.iloc[y_test.index] if hasattr(y_test, 'index') else X.iloc[-len(y_test):]
# predictions_df = pd.concat([predictions_df.reset_index(drop=True), 
#                             X_test_unscaled.reset_index(drop=True)], axis=1)

# print("===== PREDICTIONS SAMPLE =====")
# print(predictions_df[['actual_points', 'predicted_points', 'error', 'abs_error', 
#                       'overall_average', 'last_3_average', 'is_home_target']].head(10))
# print(f"\nMean Absolute Error: {predictions_df['abs_error'].mean():.3f}")
# print(f"Root Mean Squared Error: {np.sqrt((predictions_df['error']**2).mean()):.3f}")

In [57]:
# ===== FEATURE IMPORTANCE =====
# Access the fitted models from the VotingRegressor
gb_importance = model.named_estimators_['gb'].feature_importances_
rf_importance = model.named_estimators_['rf'].feature_importances_

# Average importance across models
avg_importance = (gb_importance * 2 + rf_importance * 2) / 4

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': avg_importance,
    'gb_importance': gb_importance,
    'rf_importance': rf_importance
}).sort_values('importance', ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(feature_importance_df)


===== FEATURE IMPORTANCE =====
                         feature  importance  gb_importance  rf_importance
5                  current_price    0.243502       0.349951       0.137054
18     location_adjusted_average    0.103361       0.110274       0.096448
15               recent_momentum    0.075326       0.057947       0.092704
7                  match_minus_1    0.069503       0.067223       0.071784
4                 last_2_average    0.044258       0.020660       0.067857
0                   home_average    0.043778       0.031089       0.056467
3                overall_average    0.041957       0.027551       0.056362
12                    form_trend    0.041506       0.047838       0.035173
14               price_per_point    0.032508       0.025599       0.039417
2                         rating    0.029378       0.001416       0.057340
1                   away_average    0.028226       0.019734       0.036717
10             matchup_prob_draw    0.027270       0.032863       0.

Save the model

In [58]:
import joblib

# After training your model
model_package = {
    'model': model,
    'scaler': scaler,
    'feature_columns': feature_columns
}

joblib.dump(model_package, '../data/model/fantasy_model_complete.pkl')

['../data/model/fantasy_model_complete.pkl']

In [59]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Performance Analysis', fontsize=16, fontweight='bold')

# 1. Actual vs Predicted Scatter Plot
axes[0, 0].scatter(predictions_df['actual_points'], predictions_df['predicted_points'], 
                   alpha=0.5, s=30, c='steelblue')
# Perfect prediction line
min_val = min(predictions_df['actual_points'].min(), predictions_df['predicted_points'].min())
max_val = max(predictions_df['actual_points'].max(), predictions_df['predicted_points'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Points', fontsize=11)
axes[0, 0].set_ylabel('Predicted Points', fontsize=11)
axes[0, 0].set_title('Actual vs Predicted Points', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Residual Plot (errors)
axes[0, 1].scatter(predictions_df['predicted_points'], predictions_df['error'], 
                   alpha=0.5, s=30, c='coral')
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted Points', fontsize=11)
axes[0, 1].set_ylabel('Residual (Actual - Predicted)', fontsize=11)
axes[0, 1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Error Distribution
axes[0, 2].hist(predictions_df['error'], bins=30, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 2].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0, 2].set_xlabel('Prediction Error', fontsize=11)
axes[0, 2].set_ylabel('Frequency', fontsize=11)
axes[0, 2].set_title('Error Distribution', fontsize=12, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Feature Importance (Top 10)
top_features = feature_importance_df.head(10)
axes[1, 0].barh(range(len(top_features)), top_features['importance'], color='skyblue')
axes[1, 0].set_yticks(range(len(top_features)))
axes[1, 0].set_yticklabels(top_features['feature'], fontsize=9)
axes[1, 0].set_xlabel('Importance', fontsize=11)
axes[1, 0].set_title('Top 10 Feature Importance', fontsize=12, fontweight='bold')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 5. Absolute Error vs Actual Points
axes[1, 1].scatter(predictions_df['actual_points'], predictions_df['abs_error'], 
                   alpha=0.5, s=30, c='purple')
axes[1, 1].axhline(y=predictions_df['abs_error'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean MAE: {predictions_df["abs_error"].mean():.2f}')
axes[1, 1].set_xlabel('Actual Points', fontsize=11)
axes[1, 1].set_ylabel('Absolute Error', fontsize=11)
axes[1, 1].set_title('Absolute Error vs Actual Points', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Prediction Accuracy Bins
bins = [0, 2, 5, 10, 15, 20, 30]
predictions_df['actual_bin'] = pd.cut(predictions_df['actual_points'], bins=bins)
accuracy_by_bin = predictions_df.groupby('actual_bin', observed=True)['abs_error'].mean()
axes[1, 2].bar(range(len(accuracy_by_bin)), accuracy_by_bin.values, color='salmon', alpha=0.7)
axes[1, 2].set_xticks(range(len(accuracy_by_bin)))
axes[1, 2].set_xticklabels([str(x) for x in accuracy_by_bin.index], rotation=45, ha='right', fontsize=9)
axes[1, 2].set_xlabel('Actual Points Range', fontsize=11)
axes[1, 2].set_ylabel('Mean Absolute Error', fontsize=11)
axes[1, 2].set_title('Model Accuracy by Point Range', fontsize=12, fontweight='bold')
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_performance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Additional summary statistics
print("\n===== DETAILED PERFORMANCE METRICS =====")
print(f"Mean Absolute Error (MAE): {predictions_df['abs_error'].mean():.3f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt((predictions_df['error']**2).mean()):.3f}")
print(f"Median Absolute Error: {predictions_df['abs_error'].median():.3f}")
print(f"Max Error: {predictions_df['abs_error'].max():.3f}")
print(f"Predictions within 2 points: {(predictions_df['abs_error'] <= 2).sum() / len(predictions_df) * 100:.1f}%")
print(f"Predictions within 5 points: {(predictions_df['abs_error'] <= 5).sum() / len(predictions_df) * 100:.1f}%")
print(f"\nR² Score: {model.score(X_test, y_test):.3f}")


===== DETAILED PERFORMANCE METRICS =====
Mean Absolute Error (MAE): 1.677
Root Mean Squared Error (RMSE): 2.510
Median Absolute Error: 1.036
Max Error: 12.983
Predictions within 2 points: 71.1%
Predictions within 5 points: 92.8%

R² Score: 0.364


/var/folders/r5/bx2jhb6n64n4zm91gw712nr00000gn/T/ipykernel_70666/3933795618.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [60]:
# Group highly correlated features and keep only the best from each group
correlation_matrix = df[feature_columns].corr()
correlation_matrix

,home_average,away_average,rating,overall_average,last_2_average,current_price,is_home,match_minus_1,match_minus_2,matchup_prob_win,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
home_average,1.000000,0.583046,0.712096,0.867344,0.548244,0.603933,0.025001,0.474811,0.459274,0.137089,...,0.806069,0.126863,0.130628,0.084073,0.060115,-0.007480,-0.096879,0.063185,-0.031794,0.136616
away_average,0.583046,1.000000,0.574406,0.860846,0.492488,0.506794,0.017742,0.417576,0.418487,0.088842,...,0.766115,0.085228,0.086661,0.059661,0.016111,-0.009553,-0.032985,0.035346,-0.022192,0.083363
rating,0.712096,0.574406,1.000000,0.700290,0.575887,0.842292,0.006329,0.488749,0.491131,0.193442,...,0.641576,0.182252,0.186481,0.131777,0.112038,-0.108681,-0.044313,0.055866,-0.046140,0.201349
overall_average,0.867344,0.860846,0.700290,1.000000,0.558846,0.609650,0.020444,0.479773,0.471024,0.133324,...,0.864194,0.125033,0.128143,0.069730,0.050778,-0.010458,-0.060257,0.054299,-0.034148,0.133769
last_2_average,0.548244,0.492488,0.575887,0.558846,1.000000,0.451647,0.022050,0.823947,0.833392,0.050477,...,0.507391,0.047293,0.048485,0.067088,0.048269,-0.065538,-0.036801,0.033049,0.005577,0.050495
current_price,0.603933,0.506794,0.842292,0.609650,0.451647,1.000000,0.010392,0.385069,0.387266,0.247056,...,0.551586,0.229099,0.235726,0.217826,0.117538,-0.145885,-0.062420,0.073461,-0.061170,0.280451
is_home,0.025001,0.017742,0.006329,0.020444,0.022050,0.010392,1.000000,0.010364,0.020792,0.452832,...,0.121767,0.457828,0.457304,0.081915,0.077932,0.076125,0.032106,0.943141,-0.205983,0.215823
match_minus_1,0.474811,0.417576,0.488749,0.479773,0.823947,0.385069,0.010364,1.000000,0.405559,0.035987,...,0.422517,0.033635,0.034512,0.055271,0.034186,-0.053483,-0.027726,0.023310,0.010411,0.040990
match_minus_2,0.459274,0.418487,0.491131,0.471024,0.833392,0.387266,0.020792,0.405559,1.000000,0.046080,...,0.441512,0.043472,0.044460,0.057534,0.048134,-0.061822,-0.030285,0.025893,0.001306,0.036605
matchup_prob_win,0.137089,0.088842,0.193442,0.133324,0.050477,0.247056,0.452832,0.035987,0.046080,1.000000,...,0.184470,0.988697,0.995006,0.190811,0.158886,0.165172,0.069519,0.645111,-0.579461,0.670414


In [61]:
df_clean.head(2)

,player_id,name,role,round,home_average,away_average,overall_average,current_price,matches_played,rating,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
0,51ffb2540ac2ec8b0700001e,Dani Rodríguez,centrocampista,13,2.0,0.0,2.000000,1000000,2,1,...,0.0,-0.528376,0.605592,0.0,0.160761,0.0,0.0,0.000000,1,0
1,52013ee178b20d7f07000351,Josan,centrocampista,13,4.4,1.5,3.571429,1000000,7,1,...,4.4,-0.570438,0.555189,0.0,0.150752,0.0,0.0,0.062813,1,0


In [62]:
df_clean

,player_id,name,role,round,home_average,away_average,overall_average,current_price,matches_played,rating,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
0,51ffb2540ac2ec8b0700001e,Dani Rodríguez,centrocampista,13,2.000000,0.000000,2.000000,1000000,2,1,...,0.000000,-0.528376,0.605592,0.000000,0.160761,0.000000,0.000000,0.000000,1,0
1,52013ee178b20d7f07000351,Josan,centrocampista,13,4.400000,1.500000,3.571429,1000000,7,1,...,4.400000,-0.570438,0.555189,0.000000,0.150752,0.000000,0.000000,0.062813,1,0
2,521a999b5865e5687000001c,David Soria,portero,13,7.571429,3.750000,5.533333,20975958,15,5,...,7.571429,-0.355844,0.828432,0.000000,0.000000,0.000000,0.689820,0.092138,1,0
3,5203972e2e80d2950b00016f,J. Musso,portero,13,0.000000,4.000000,4.000000,1000000,1,1,...,4.000000,0.355844,1.895964,0.000000,0.000000,0.000000,1.223586,0.000000,0,1
4,57363a66ad212396073bce9f,Zubeldia,defensa,13,4.000000,3.666667,3.833333,5422151,12,3,...,3.666667,0.008957,1.360839,0.000000,0.000000,0.504479,0.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4160,696a39e9af58fa040efb5372,Kalumba,centrocampista,20,0.000000,0.000000,0.000000,1000000,0,1,...,0.000000,0.013468,1.415417,0.000000,0.482339,0.000000,0.000000,0.200975,0,0
4161,66a3d3797fc6ff03fb06facf,Boselli,defensa,20,0.000000,0.000000,0.000000,2249716,0,1,...,0.000000,0.105420,1.554784,0.000000,0.000000,0.552710,0.000000,0.224682,0,0
4162,612183f216ab17088eaf36b1,Satriano,delantero,20,3.000000,0.000000,3.000000,4852987,1,3,...,3.000000,0.105420,1.554784,0.898728,0.000000,0.000000,0.000000,0.224682,0,0
4163,696c1bb8af58fa040efb6d15,Zaid Romero,defensa,20,0.000000,0.000000,0.000000,1749991,0,1,...,0.000000,0.105420,1.554784,0.000000,0.000000,0.552710,0.000000,0.224682,0,0
